# Optional Deep-Dive: How Attention Works (Pure Python)

> OPTIONAL SUPPLEMENTARY NOTEBOOK
>
> This is an optional deep-dive. The main course path does NOT require it.
> The required path teaches the attention CONCEPT as a short mini-lesson so you
> can read attention-head visualisations and understand CLS tokens. THIS notebook
> is the from-scratch BUILD: you implement the mechanism yourself in NumPy.
> You can complete the course without ever opening this notebook.

## Who this notebook is for

You should work through this notebook if you want to understand what is
happening INSIDE an attention layer, not just how to call one. It is aimed at
learners who are comfortable with NumPy and want to see attention built from
first principles.

If you only care about USING attention-based models (calling an LLM, fine-tuning
a Transformer, prompting), you can safely skip this notebook. The main course
covers the attention concept you need as a user.

## This notebook is self-contained

Everything you need is defined inside this notebook. It does not depend on any
other course notebook, and you do not need to have run any earlier topic. You
can open it cold and run it start to finish.

It also runs OFFLINE. The first demo would normally download a small word2vec
sample, but if there is no network the notebook falls back to a tiny set of
embedding vectors defined inline, so every cell still runs.

This notebook writes no files to disk. It produces only inline plots.

There is a companion optional notebook that ports these same ideas to PyTorch.
That companion is also optional. You may read it after this one if you want to
see the framework version, but it is not a required next step.

## What you will build

By the end you will have implemented, from scratch in NumPy:
- A simulation of seq2seq without attention, and seen exactly where it breaks down
- Bahdanau (additive) attention: alignment scores, softmax weights, context vector
- Dot product attention: a simpler variant of the same idea
- Scaled dot product attention: the version used in Transformers

## Learning objectives

1. Explain the fixed-size bottleneck problem in seq2seq models
2. Implement the Bahdanau attention score computation step by step
3. Compare additive vs dot product vs scaled dot product attention
4. Visualise attention weights as a heatmap over complaint tokens

## Before the math: why should a user care about attention?

Attention is the single idea that made modern language models work. Every model
you use as a developer -- the LLM behind a chat API, a Transformer you fine-tune,
a translation model -- is built out of attention layers.

As a user you mostly do not need to know the internals. But three things that you
DO notice every day come straight out of how attention works:

1. Context windows. A model can "look back" at any earlier token in the prompt
   because attention lets every position read every other position directly.
   The size of that window is bounded by how much attention you can afford to
   compute.

2. Long-input quality. Older sequence models compressed an entire input into one
   fixed-size vector, so details early in a long input got washed out. Attention
   removed that bottleneck. This is why a modern model can answer a question about
   page 1 of a long document you pasted on page 40.

3. Cost and latency. Attention compares every token to every other token, so its
   cost grows with the square of the input length. That is the real reason long
   prompts are slower and more expensive.

This notebook opens up that mechanism. We build it in plain NumPy on a small
customer-complaint example so you can see every number. You will not need this to
USE a model, but once you have seen it, the behaviour above stops being magic.

## The running example: a customer-complaint router

To keep every number concrete we use one running example throughout this
notebook: an imagined customer-support team that receives thousands of complaint
messages a day and wants to route them automatically.

Long complaints (50 or more tokens) are the hardest to route, because they
contain multiple issues. A plain seq2seq model compresses the entire complaint
into one fixed-size vector, so important details at the end of a long message get
lost. Attention fixes this.

You do not need any prior course notebook to follow along. The next cells install
the packages and define the small vocabulary this notebook uses.

In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Environment setup for SageMaker Studio
# All attention demos run in this kernel - no remote training jobs in this notebook.

!pip install -q "sagemaker>=2.200.0,<3.0.0" \
    "gensim>=4.3.0,<5.0.0" \
    "nltk>=3.8.0" \
    "numpy<2" \
    "matplotlib>=3.7.0" \
    "seaborn>=0.12.0"

print("RESTART KERNEL before continuing -- environment packages were installed/upgraded.")


In [ ]:
import sagemaker
from sagemaker import get_execution_role
import boto3

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print("Environment ready.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re

# seaborn is optional - it only makes the heatmaps prettier. A locked-down
# classroom image may not have it, so we guard the import. Codex R2 finding N1.
try:
    import seaborn as sns
    _HAS_SEABORN = True
except ImportError:
    sns = None
    _HAS_SEABORN = False
    print("seaborn not installed - heatmaps will use plain matplotlib.")

# This notebook can run fully OFFLINE. We try to load the small NLTK word2vec
# sample for nicer real-word embeddings, but if there is no network (a common
# case on a locked-down classroom image) we fall back to a tiny inline table of
# random-but-fixed embedding vectors. Either way every demo below still runs.

EMBED_DIM = 300          # word2vec sample dimension; the inline fallback matches it
_WORD2VEC_MODEL = None   # populated lazily if the real model is available
_WORD2VEC_AVAILABLE = False

# Every word this notebook ever asks to embed. Keeping this list explicit means
# the offline fallback can guarantee an embedding for all of them.
_NOTEBOOK_VOCAB = [
    "charge", "unauthorised", "unauthorized", "account", "refund", "dispute",
    "urgent", "contacted", "branch", "twice", "no", "response", "still",
    "waiting", "three", "weeks", "unable", "resolve", "issue", "extremely",
    "frustrated", "customer", "fraud", "help", "transaction", "card", "blocked",
    "payment", "failed", "interest", "disputed", "transfer", "balance",
    "declined", "overdraft",
]

# Inline fallback: deterministic pseudo-embeddings, one per vocab word.
# A fixed seed makes the fallback reproducible run to run.
_fallback_rng = np.random.RandomState(20240517)
_INLINE_EMBEDDINGS = {
    word: _fallback_rng.randn(EMBED_DIM).astype(np.float64)
    for word in _NOTEBOOK_VOCAB
}

try:
    import nltk
    from nltk.data import find
    import gensim
    nltk.download("word2vec_sample", quiet=True)
    _word2vec_path = str(find("models/word2vec_sample/pruned.word2vec.txt"))
    _WORD2VEC_MODEL = gensim.models.KeyedVectors.load_word2vec_format(
        _word2vec_path, binary=False
    )
    _WORD2VEC_AVAILABLE = True
    print("word2vec sample loaded - using real pretrained embeddings.")
except Exception as exc:
    print("word2vec sample is not available (offline or download blocked).")
    print(f"Reason: {exc}")
    print("Falling back to the inline embedding table. Every demo still runs.")

def softmax(x, axis=-1):
    """Compute softmax along the specified axis. Numerically stable version."""
    # Subtract max for numerical stability before exp
    x = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def get_word2vec_embedding(words):
    """
    Return embeddings for the given words.

    If the NLTK word2vec sample loaded successfully, real pretrained vectors are
    used. Otherwise the inline fallback table is used so the notebook runs
    offline. Words with no available vector are silently skipped.

    This function ALWAYS returns a usable result: as long as at least one input
    word is in the notebook vocabulary it returns a non-empty embedding array,
    so downstream cells never see an empty array or an undefined variable.

    Returns:
        embeddings: array of shape (N_found, EMBED_DIM)
        words_pass: the list of input words that were embedded, same order
    """
    output = []
    words_pass = []
    for word in words:
        vec = None
        if _WORD2VEC_AVAILABLE:
            try:
                vec = np.array(_WORD2VEC_MODEL.get_vector(word), dtype=np.float64)
            except KeyError:
                vec = None
        if vec is None:
            # Offline fallback (also covers words missing from word2vec).
            vec = _INLINE_EMBEDDINGS.get(word)
        if vec is not None:
            output.append(vec)
            words_pass.append(word)
    embeddings = np.array(output, dtype=np.float64)
    return embeddings, words_pass

def plot_attention_weight_matrix(weight_matrix, x_ticks, y_ticks, title="Attention weights"):
    """Plot a 2D attention weight matrix as a heatmap.

    Uses seaborn if available, otherwise plain matplotlib (Codex R2 finding N1).
    """
    plt.figure(figsize=(max(8, len(x_ticks) * 1.2), max(5, len(y_ticks) * 0.8)))
    if _HAS_SEABORN:
        sns.heatmap(weight_matrix, cmap="Blues", annot=True, fmt=".2f",
                    linewidths=0.5, linecolor="lightgray")
    else:
        plt.imshow(weight_matrix, cmap="Blues", aspect="auto")
        plt.colorbar()
        for r in range(weight_matrix.shape[0]):
            for c in range(weight_matrix.shape[1]):
                plt.text(c, r, f"{weight_matrix[r, c]:.2f}", ha="center", va="center")
    plt.xticks(np.arange(weight_matrix.shape[1]) + 0.5, x_ticks, rotation=30, ha="right")
    plt.yticks(np.arange(weight_matrix.shape[0]) + 0.5, y_ticks, rotation=0)
    plt.title(title)
    plt.xlabel("Key tokens")
    plt.ylabel("Query tokens")
    plt.tight_layout()
    plt.show()

print("Imports and helpers loaded.")

In [ ]:
# Complaint token vocabulary for this notebook.
# This notebook is standalone, so we define the vocabulary right here.
# Other course notebooks use a similar list; we keep a local copy so this
# notebook runs on its own with no dependency on any earlier topic.
COMPLAINT_TOKENS = [
    "unauthorized", "transaction", "account", "card", "blocked",
    "payment", "failed", "interest", "charge", "disputed",
    "transfer", "balance", "fraud", "declined", "overdraft"
]
COMPLAINT_LABELS = ["fraud", "billing", "access", "payment", "general"]
print(f"Complaint vocabulary: {len(COMPLAINT_TOKENS)} tokens, {len(COMPLAINT_LABELS)} labels")

## Section 1 - The Fixed-Size Bottleneck Problem

### The situation in our running example

Imagine the complaints routing pipeline uses a seq2seq model: an LSTM encoder
reads the full complaint message and compresses it into a single hidden state
vector. The decoder then generates a category label from that one vector.

This works for short messages. But watch what happens with a long complaint.

In [ ]:
# Beat 1: The seq2seq bottleneck problem in action.
# We simulate what a trained LSTM encoder does: compress a sequence of word vectors
# into a single fixed-size context vector by taking the LAST hidden state.
#
# Watch what happens as the complaint gets longer.

import numpy as np

def simulate_encoder_no_attention(word_embeddings, hidden_size=64):
    """
    Simulate a (very simplified) LSTM encoder that compresses all word embeddings
    into a single hidden state by averaging progressively - mimicking how information
    from early words gets diluted as the sequence grows.

    This is NOT a real LSTM, but it reproduces the bottleneck problem faithfully:
    the fixed-size output cannot carry all information from a long sequence.
    """
    # Simulate hidden state update: weighted running average that decays early tokens
    # As sequence length grows, early tokens contribute less and less.
    T = word_embeddings.shape[0]
    d = word_embeddings.shape[1]

    # Truncate or project to hidden_size
    projection = np.random.randn(d, hidden_size) * 0.1
    hidden = np.zeros(hidden_size)

    for t in range(T):
        token_vec = word_embeddings[t] @ projection  # shape: (hidden_size,)
        # Decay factor: each new token dilutes the previous hidden state
        decay = 0.85  # simulates vanishing gradient / info compression
        hidden = decay * hidden + (1 - decay) * token_vec

    return hidden  # single fixed-size vector: the bottleneck

# Simulate two complaints:
# Short complaint (6 tokens)
short_complaint = "charge unauthorised account refund dispute urgent"
# Long complaint (20 tokens) - starts with the same urgent issue but has more detail
long_complaint = (
    "charge unauthorised account refund dispute urgent "
    "contacted branch twice no response still waiting three weeks "
    "unable resolve issue extremely frustrated customer"
)

np.random.seed(42)

short_words = short_complaint.split()
long_words = long_complaint.split()

short_embeddings, short_found = get_word2vec_embedding(short_words)
long_embeddings, long_found = get_word2vec_embedding(long_words)

print(f"Short complaint: {len(short_found)} tokens embedded")
print(f"Long complaint:  {len(long_found)} tokens embedded")

short_context = simulate_encoder_no_attention(short_embeddings)
long_context_first6 = simulate_encoder_no_attention(long_embeddings[:6])  # same first 6 tokens
long_context_full = simulate_encoder_no_attention(long_embeddings)

# How similar is the long-complaint context vector to the short one?
# If seq2seq worked well, these should be very similar (same first 6 tokens).
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

sim_short_vs_long6 = cosine_similarity(short_context, long_context_first6)
sim_short_vs_long_full = cosine_similarity(short_context, long_context_full)

print(f"\nCosine similarity (short vs long[first 6 same tokens]):  {sim_short_vs_long6:.4f}")
print(f"Cosine similarity (short vs long[full 20 tokens]):       {sim_short_vs_long_full:.4f}")
print()
print("PROBLEM: Even though the first 6 tokens are identical, the encoder produces")
print("different context vectors depending on what comes after them.")
print("The fixed-size bottleneck cannot preserve all the information.")
print()
print("Now imagine trying to triage a 50-token complaint from a fixed 64-dim vector.")
print("Early details about 'unauthorised charge' get washed out by later tokens.")

### Discussion (3 minutes)

Think about this from a Barclays operations perspective:

1. A complaint reads: "I noticed an unauthorised charge on my account last Tuesday.
   I tried calling the helpline three times but was put on hold each time for over
   45 minutes. This is completely unacceptable and I want a full refund immediately."

   If the routing model sees only ONE context vector from this -- what information
   is most at risk of being lost? What routing decision could go wrong?

2. In production, long complaints correlate with high-severity cases (customers who
   are very frustrated write more). If our model systematically fails on long
   complaints, what is the business impact?

3. What would you want the model to do differently?

In [ ]:
# Beat 2: Visual anchor for the bottleneck vs attention concept.
# The diagram below shows exactly what changes when we add attention.
print("See diagram: seq2seq fixed bottleneck vs Bahdanau attention")

<!-- DIAGRAM: seq2seq encoder compressing a long complaint into a single fixed-size bottleneck vector on the left, versus seq2seq with Bahdanau attention showing decoder attending to all encoder hidden states via weighted alignment arrows on the right -->

```mermaid
graph TD
    subgraph bottleneck ["Seq2seq WITHOUT attention"]
        enc1["Encoder reads\nfull complaint"] --> bottle["Single bottleneck vector\n(fixed size, all info compressed)"]
        bottle --> dec1["Decoder generates\noutput from bottleneck only"]
    end

    subgraph attention ["Seq2seq WITH Bahdanau attention"]
        enc2["Encoder produces\nh1, h2, ..., hT\n(one state per token)"]
        dec2["Decoder state s_t-1"] --> align["Alignment scores\n(how much to attend to each h)"]
        enc2 --> align
        align --> ctx["Context vector c_t\n(weighted sum of encoder states)"]
        ctx --> dec3["Decoder generates\nnext output token"]
    end

    style bottle fill:#fda,stroke:#c33
    style ctx fill:#dfd,stroke:#3a3
```

*Figure: The left side shows the bottleneck: ALL the complaint information must pass through one vector. The right side shows Bahdanau attention: at each decoder step, we can look back at any encoder hidden state. No single bottleneck.*

In [ ]:
# Beat 3: Attention SOLVES the bottleneck.
# We show how attention lets the decoder produce a DIFFERENT context vector at
# each step by attending to ALL encoder outputs (not just the last hidden state).
#
# This is a preview: the full math arrives in Section 2. For now, just observe
# that each decoder step gets its own weighted blend of encoder outputs.

np.random.seed(123)

# Simulate 6 encoder outputs (one per complaint token)
preview_tokens = ["charge", "unauthorised", "account", "refund", "dispute", "urgent"]
preview_emb, preview_found = get_word2vec_embedding(preview_tokens)
T_preview = preview_emb.shape[0]
d_preview = preview_emb.shape[1]

# Simulate 3 decoder states (3 output steps being generated)
decoder_steps = np.random.randn(3, d_preview)

# Compute attention weights for each decoder step via simple dot product + softmax.
# (This is dot product attention -- the simplest variant. Bahdanau adds learned
#  weights on top; we will see that in Section 2.)
all_alpha = []
all_contexts = []
for step in range(3):
    s = decoder_steps[step]
    scores = preview_emb @ s            # (T_preview,)
    alpha_step = softmax(scores)        # attention weights for this decoder step
    context_step = alpha_step @ preview_emb  # weighted sum of encoder outputs
    all_alpha.append(alpha_step)
    all_contexts.append(context_step)

print("Attention weights per decoder step (rows = decoder step, cols = complaint tokens):")
print(f"{'step':>6}  " + "  ".join(f"{t:>13s}" for t in preview_found))
for i, a in enumerate(all_alpha):
    print(f"{i:>6}  " + "  ".join(f"{w:>13.4f}" for w in a))

print()
print("Notice: each decoder step produces a DIFFERENT attention distribution.")
print("No single bottleneck -- the decoder can look at any encoder position it needs.")
print()

# Confirm the context vectors actually differ across decoder steps
diff_01 = float(np.linalg.norm(all_contexts[0] - all_contexts[1]))
diff_02 = float(np.linalg.norm(all_contexts[0] - all_contexts[2]))
print(f"||context_0 - context_1|| = {diff_01:.4f}")
print(f"||context_0 - context_2|| = {diff_02:.4f}")
print("These are non-zero, which means the decoder reads different summaries each step.")


### Quick Observation (2 minutes, no coding)

Look at the attention weights printed above. Answer these in your notes or with a neighbour:

1. For decoder step 1, which complaint token receives the highest attention weight? Why might that token be the most useful one to focus on when routing a complaint?

2. Do the three decoder steps attend to the same token, or different tokens? What does that tell you about how a seq2seq model with attention behaves compared to one without?

3. In a Barclays complaint routing system, why is it valuable that the model can attend differently at each output step (for example, when generating a category like "fraud" versus an urgency tag like "high")?

You will implement this attention mechanism properly in Lab 1.

## Section 2 - Bahdanau Attention: The Math

Bahdanau et al. (2015) introduced "additive attention". The core idea:

At each decoder step t, instead of reading one fixed context vector,
we compute a DIFFERENT context vector by weighting all encoder hidden states.

Step 1 - Alignment scores (how relevant is encoder state h_i to decoder state s_{t-1}):

    e_{t,i} = v^T * tanh(W1 * h_i + W2 * s_{t-1})

Step 2 - Attention weights (normalise so they sum to 1):

    alpha_{t,i} = softmax_i(e_{t,i})

Step 3 - Context vector (weighted sum of all encoder states):

    c_t = sum_i( alpha_{t,i} * h_i )

The weight alpha_{t,i} answers: "When decoding token t, how much should I look at
encoder position i?" A high weight means position i is important for this step.

In our complaint routing case: when deciding "is this a fraud complaint?",
the model should attend strongly to tokens like "unauthorised", "charge", "fraud"
regardless of where they appear in the message.

In [ ]:
# Beat 2: Bahdanau score computation diagram anchor.
# The diagram below traces the full computation for one decoder step.
print("See diagram: Bahdanau attention score computation step by step")

<!-- DIAGRAM: Bahdanau attention score computation step by step: encoder hidden states h1 through hT on the left; decoder previous state s_{t-1} on the right; W1 and W2 projection arrows converging to a tanh node; v scoring vector producing energy scalars e_{t,1} through e_{t,T}; softmax normalisation producing alpha weights; weighted sum arrow pointing to context vector c_t. All tensor shapes labelled. -->

```mermaid
graph TD
    h["Encoder hidden states\nh1...hT  [T x enc_dim]"] --> w1["W1 projection\n[enc_dim -> attn_dim]"]
    s["Decoder prev state\ns_t-1  [dec_dim]"] --> w2["W2 projection\n[dec_dim -> attn_dim]"]
    w1 --> tanh["tanh(W1*h + W2*s)\n[T x attn_dim]"]
    w2 --> tanh
    tanh --> v["v dot product\n(learned vector [attn_dim])"]
    v --> energy["Energy scalars\ne_t,1 ... e_t,T  [T]"]
    energy --> softmax["Softmax\nnormalise over T positions"]
    softmax --> alpha["Attention weights\nalpha_t,1 ... alpha_t,T  [T]"]
    alpha --> ctx["Context vector c_t\nweighted sum of h1...hT  [enc_dim]"]
    h --> ctx

    style alpha fill:#ffd,stroke:#aa3
    style ctx fill:#dfd,stroke:#3a3
    style s fill:#ddf,stroke:#33c
```

*Figure: The diagram traces the computation for ONE decoder step t. Notice that the alignment network (W1, W2, v) is shared across all decoder steps -- the same weights compute alignment scores at every position. Only the decoder state s_{t-1} changes per step.*

In [ ]:
# Beat 3: Full working implementation of Bahdanau (additive) attention in NumPy.
# Instructor: walk through each step slowly. Students should read every comment.
#
# We use small dimensions so shapes are visible:
# - T_enc = number of encoder time steps (complaint tokens)
# - T_dec = number of decoder steps (we do one step to keep it clear)
# - d_h = encoder hidden size
# - d_s = decoder hidden size
# - d_align = alignment hidden layer size

np.random.seed(0)

# Dimensions (small for classroom clarity)
T_enc = 8     # 8 complaint tokens in the encoder
d_h = 16      # encoder hidden state dimension
d_s = 16      # decoder hidden state dimension
d_align = 8   # Bahdanau alignment hidden layer size

# --- Learnable weights (in a real model these are trained) ---
# W1 maps encoder hidden states to alignment space: shape (d_align, d_h)
W1 = np.random.randn(d_align, d_h) * 0.1
# W2 maps decoder hidden state to alignment space: shape (d_align, d_s)
W2 = np.random.randn(d_align, d_s) * 0.1
# v is the alignment scoring vector: shape (d_align,)
v = np.random.randn(d_align) * 0.1

def bahdanau_attention(encoder_states, decoder_state, W1, W2, v):
    """
    Compute one step of Bahdanau (additive) attention.

    Args:
        encoder_states: shape (T_enc, d_h) - all encoder hidden states
        decoder_state:  shape (d_s,) - current decoder hidden state s_{t-1}
        W1:             shape (d_align, d_h) - encoder weight matrix
        W2:             shape (d_align, d_s) - decoder weight matrix
        v:              shape (d_align,) - alignment scoring vector

    Returns:
        context_vector: shape (d_h,) - weighted sum of encoder states
        alpha:          shape (T_enc,) - attention weights (sum to 1)
        energy:         shape (T_enc,) - raw alignment scores before softmax
    """
    T_enc = encoder_states.shape[0]

    # Step 1: Compute alignment scores for each encoder position.
    # For each encoder position i, score how relevant h_i is to s_{t-1}.
    energy = np.zeros(T_enc)
    for i in range(T_enc):
        h_i = encoder_states[i]          # shape: (d_h,)
        # Project encoder state to alignment space
        enc_part = W1 @ h_i              # shape: (d_align,)
        # Project decoder state to alignment space
        dec_part = W2 @ decoder_state    # shape: (d_align,)
        # Combine with tanh nonlinearity and score with v
        energy[i] = v @ np.tanh(enc_part + dec_part)  # scalar

    # Step 2: Softmax over all encoder positions -> attention weights alpha
    # alpha[i] = "how much to attend to encoder position i at this decoder step"
    alpha = softmax(energy)  # shape: (T_enc,) - sums to 1.0

    # Step 3: Weighted sum of encoder states = context vector
    # context is now a DIFFERENT summary for each decoder step, not one fixed vector
    context_vector = np.zeros(d_h)
    for i in range(T_enc):
        context_vector += alpha[i] * encoder_states[i]
    # Equivalently (vectorised): context_vector = alpha @ encoder_states

    return context_vector, alpha, energy

# --- Demonstration ---
# Simulate 8 encoder hidden states for a complaint about "unauthorised charge"
# In a real model these come from an LSTM; here we use word embeddings directly.
complaint_tokens = ["unauthorised", "charge", "account", "refund", "dispute", "urgent", "contacted", "branch"]
complaint_embeddings, found_tokens = get_word2vec_embedding(complaint_tokens)

# If word2vec did not find all tokens, pad with zeros for the demo
T_found = len(found_tokens)
if T_found < T_enc:
    padding = np.zeros((T_enc - T_found, complaint_embeddings.shape[1]))
    complaint_embeddings = np.vstack([complaint_embeddings, padding])
    found_tokens = found_tokens + complaint_tokens[T_found:]

# Project down to d_h via a random projection (simulating encoder LSTM output)
proj_enc = np.random.randn(complaint_embeddings.shape[1], d_h) * 0.1
encoder_states = complaint_embeddings @ proj_enc  # shape: (T_enc, d_h)

# Simulate a decoder state asking "is this about fraud?"
# (In a real decoder this is the previous LSTM hidden state)
decoder_state = np.random.randn(d_s)

# Run attention
context, alpha, energy = bahdanau_attention(encoder_states, decoder_state, W1, W2, v)

print("Bahdanau Attention Demo")
print("=" * 40)
print(f"Encoder states shape: {encoder_states.shape}  -> ({T_enc} tokens, {d_h} dims)")
print(f"Decoder state shape:  {decoder_state.shape}  -> ({d_s} dims)")
print(f"Context vector shape: {context.shape}         -> ({d_h} dims)")
print()
print("Attention weights (alpha) per complaint token:")
for tok, a in zip(found_tokens, alpha):
    bar = "#" * int(a * 50)
    print(f"  {tok:20s}  {a:.4f}  {bar}")
print(f"\nalpha sums to: {alpha.sum():.6f} (must be 1.0)")
print()
print("Key insight: the context vector is a DIFFERENT weighted blend at each decoder step.")
print("The decoder can focus on 'unauthorised' and 'charge' when routing to fraud,")
print("regardless of where those tokens appear in the message.")

In [ ]:
# Beat 3 (continued): Vectorised Bahdanau attention and attention heatmap.
#
# For a full sequence of decoder steps we want to compute attention in parallel.
# Here we run multiple decoder steps and build the full attention weight matrix.

def bahdanau_attention_vectorised(encoder_states, decoder_states, W1, W2, v):
    """
    Compute Bahdanau attention for all decoder steps at once.

    Args:
        encoder_states:  shape (T_enc, d_h)
        decoder_states:  shape (T_dec, d_s)
        W1, W2, v:       alignment parameters (same as before)

    Returns:
        context_vectors: shape (T_dec, d_h)
        alpha_matrix:    shape (T_dec, T_enc) - full attention weight matrix
    """
    T_enc = encoder_states.shape[0]
    T_dec = decoder_states.shape[0]

    # Project ALL encoder states at once: (T_enc, d_align)
    enc_projected = encoder_states @ W1.T   # (T_enc, d_align)
    # Project ALL decoder states at once: (T_dec, d_align)
    dec_projected = decoder_states @ W2.T   # (T_dec, d_align)

    # Broadcast: enc_projected[None, :, :] + dec_projected[:, None, :]
    # -> shape (T_dec, T_enc, d_align)
    combined = enc_projected[np.newaxis, :, :] + dec_projected[:, np.newaxis, :]
    # Apply tanh then score with v -> shape (T_dec, T_enc)
    energy = np.tanh(combined) @ v           # (T_dec, T_enc)

    # Softmax over encoder positions (axis=-1 means over T_enc)
    alpha_matrix = softmax(energy, axis=-1)  # (T_dec, T_enc)

    # Context vectors: (T_dec, T_enc) x (T_enc, d_h) -> (T_dec, d_h)
    context_vectors = alpha_matrix @ encoder_states

    return context_vectors, alpha_matrix

# Simulate 5 decoder steps (5 output tokens being generated)
T_dec = 5
np.random.seed(7)
decoder_states = np.random.randn(T_dec, d_s)

contexts, alpha_mat = bahdanau_attention_vectorised(encoder_states, decoder_states, W1, W2, v)

print(f"Alpha matrix shape: {alpha_mat.shape}  -> ({T_dec} decoder steps, {T_enc} encoder positions)")
print(f"Context vectors shape: {contexts.shape}")

# Show the attention heatmap: rows = decoder steps, cols = complaint tokens
decoder_step_labels = [f"step_{i}" for i in range(T_dec)]
plot_attention_weight_matrix(alpha_mat, found_tokens, decoder_step_labels,
                             title="Bahdanau Attention Weights: Complaint Tokens")
print("Each row is a decoder step. Brighter cells = more attention on that complaint token.")
print("With random weights these look uniform - training would make them selective.")

## Lab 1 - Implement Bahdanau Attention Step by Step (Tier 1 - Guided)

**Time**: 15-20 minutes

### Situation

The Barclays complaints team has given you a set of encoder hidden states
representing the tokenised version of a complaint message. You have a single
decoder state representing "what we are looking for" (e.g., urgency signals).

### Task

Implement the three steps of Bahdanau attention yourself, using only NumPy.
You will compute alignment scores, attention weights, and the context vector.

### Action

Complete the function below. Each step is clearly labelled.

### Result

Your function should produce the same output as `bahdanau_attention` from the demo.
The verification cell will compare your context vector and alpha values.

In [ ]:
# Lab 1: Implement Bahdanau attention from the three-step algorithm.
# Complete the three stubs below. Use the demo (Cell 11) as your reference.

def my_bahdanau_attention(encoder_states, decoder_state, W1, W2, v):
    """
    Compute one step of Bahdanau attention.

    Args:
        encoder_states: shape (T_enc, d_h)
        decoder_state:  shape (d_s,)
        W1:             shape (d_align, d_h)
        W2:             shape (d_align, d_s)
        v:              shape (d_align,)

    Returns:
        context_vector: shape (d_h,)
        alpha:          shape (T_enc,) - sums to 1
        energy:         shape (T_enc,) - raw scores
    """
    T_enc = encoder_states.shape[0]
    energy = np.zeros(T_enc)

    for i in range(T_enc):
        h_i = encoder_states[i]

        # Step 1: Compute the alignment energy for position i.
        # enc_part = W1 applied to h_i
        # dec_part = W2 applied to decoder_state
        # energy[i] = v dot tanh(enc_part + dec_part)
        enc_part = None  # YOUR CODE
        dec_part = None  # YOUR CODE
        energy[i] = None  # YOUR CODE

    # Step 2: Convert energy to attention weights using softmax.
    alpha = None  # YOUR CODE

    # Step 3: Compute the context vector as a weighted sum of encoder states.
    context_vector = None  # YOUR CODE

    return context_vector, alpha, energy

# Quick test (run this before the verification cell)
test_ctx, test_alpha, test_energy = my_bahdanau_attention(
    encoder_states, decoder_state, W1, W2, v
)
print(f"context shape: {test_ctx.shape if test_ctx is not None else 'None - not implemented yet'}")
print(f"alpha shape:   {test_alpha.shape if test_alpha is not None else 'None - not implemented yet'}")

In [ ]:
# Lab 1 Verification - run this after completing my_bahdanau_attention above.

ref_ctx, ref_alpha, ref_energy = bahdanau_attention(encoder_states, decoder_state, W1, W2, v)
student_ctx, student_alpha, student_energy = my_bahdanau_attention(
    encoder_states, decoder_state, W1, W2, v
)

all_pass = True

if student_ctx is None or student_alpha is None:
    print("FAIL: Some return values are still None. Complete all three steps.")
    all_pass = False
else:
    # Check alpha sums to 1
    alpha_sum = float(np.sum(student_alpha))
    if abs(alpha_sum - 1.0) < 1e-5:
        print(f"PASS: alpha sums to 1.0 (got {alpha_sum:.6f})")
    else:
        print(f"FAIL: alpha should sum to 1.0, got {alpha_sum:.6f}")
        all_pass = False

    # Check context vector matches reference
    if np.allclose(student_ctx, ref_ctx, atol=1e-5):
        print("PASS: context vector matches reference implementation")
    else:
        max_diff = float(np.max(np.abs(student_ctx - ref_ctx)))
        print(f"FAIL: context vector differs from reference (max diff: {max_diff:.6f})")
        all_pass = False

    # Check alpha matches reference
    if np.allclose(student_alpha, ref_alpha, atol=1e-5):
        print("PASS: attention weights match reference implementation")
    else:
        print("FAIL: attention weights do not match reference")
        all_pass = False

if all_pass:
    print()
    print("All checks passed. Your Bahdanau attention implementation is correct.")

In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
if 'my_bahdanau_attention' not in dir() or my_bahdanau_attention(
        encoder_states, decoder_state, W1, W2, v)[0] is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    my_bahdanau_attention = bahdanau_attention

### Stretch (fast finishers)

Visualise YOUR attention weights as a heatmap over the complaint tokens.
Use `plot_attention_weight_matrix` from the demo. Does your heatmap match the
reference implementation? Try different decoder states and explain how the
attention pattern changes.

```python
# Stretch: visualise your attention weights
# ctx, my_alpha, _ = my_bahdanau_attention(encoder_states, decoder_state, W1, W2, v)
# plot_attention_weight_matrix(my_alpha.reshape(1, -1), found_tokens, ["decoder step 0"],
#                              title="My Bahdanau Attention Weights")
```

### Homework Extension

Implement Bahdanau attention with NO scaffold -- just the equation:

    e_{t,i} = v^T * tanh(W1 * h_i + W2 * s_{t-1})
    alpha = softmax(e)
    c_t = sum_i(alpha_i * h_i)

Write a function `bahdanau_from_equation(encoder_states, decoder_state, W1, W2, v)`
from scratch, without looking at the demo. Verify it produces the same result as the
verification cell above.

## Section 3 - Dot Product Attention: A Simpler Variant

Bahdanau attention requires three learned weight matrices (W1, W2, v).
What if the encoder and decoder are the same size? We can simplify:

    score(h_i, s) = h_i^T * s   (dot product, no learned weights!)

This is Luong-style or "dot product" attention. Faster, fewer parameters,
but requires encoder and decoder to share the same hidden dimension.

Let us implement it and compare the attention patterns to Bahdanau.

In [ ]:
# Beat 1: Naive dot product attention - breaks if dimensions do not match.
# Students see the error before seeing the working version.

def dot_product_attention_naive(encoder_states, decoder_state):
    """
    Naive dot product attention.
    Works ONLY if encoder_states.shape[-1] == decoder_state.shape[-1].
    """
    # This will fail if dimensions differ
    scores = encoder_states @ decoder_state   # shape: (T_enc,)
    alpha = softmax(scores)
    context = alpha @ encoder_states
    return context, alpha

# Case 1: Works fine - same dimensions
enc_same = np.random.randn(8, 16)
dec_same = np.random.randn(16)
ctx_ok, alpha_ok = dot_product_attention_naive(enc_same, dec_same)
print(f"Case 1 (same dims): context shape = {ctx_ok.shape}  -- OK")

# Case 2: Breaks - encoder has 300 dims (word2vec), decoder has 64 dims
enc_diff = np.random.randn(8, 300)   # word2vec embeddings
dec_diff = np.random.randn(64)        # smaller decoder state
try:
    ctx_fail, alpha_fail = dot_product_attention_naive(enc_diff, dec_diff)
    print(f"Case 2 (diff dims): context shape = {ctx_fail.shape}  -- this should not print")
except ValueError as e:
    print(f"Case 2 (diff dims): FAILS with ValueError: {e}")
    print()
    print("This is why Bahdanau uses W1, W2 to project to a common alignment space.")
    print("Dot product attention requires d_encoder == d_decoder.")

### Beat 2 - How Dot Product Attention Works

The dot product attention mechanism avoids the three learned weight matrices of Bahdanau
by using the encoder and decoder vectors directly:

    score(h_i, s) = h_i^T * s   (a single dot product per encoder position)

The computation graph is:

    encoder state h_i  ---\
                           dot product  -->  score e_i  -->  softmax  -->  alpha_i
    decoder state s    ---/

    context c = sum_i( alpha_i * h_i )

This is faster and has fewer parameters than Bahdanau. The trade-off is the
dimension constraint: h_i and s must have the same length, or you need projection
matrices to align them first. The Transformer solves this with learned Q, K, V
projection matrices applied before every dot product operation.

In [ ]:
# Beat 3: Working dot product attention with projection to handle dimension mismatch.
# In practice, Q, K, V projections in the Transformer handle this.

def dot_product_attention(encoder_states, decoder_state, proj_enc=None, proj_dec=None):
    """
    Dot product attention. If encoder and decoder have different dimensions,
    optional projection matrices can align them.

    Args:
        encoder_states: shape (T_enc, d_enc)
        decoder_state:  shape (d_dec,)
        proj_enc:       optional (d_enc, d_common) projection for encoder
        proj_dec:       optional (d_dec, d_common) projection for decoder

    Returns:
        context: shape (d_enc,) - weighted sum in ORIGINAL encoder space
        alpha:   shape (T_enc,) - attention weights
    """
    if proj_enc is not None:
        enc_projected = encoder_states @ proj_enc   # (T_enc, d_common)
    else:
        enc_projected = encoder_states

    if proj_dec is not None:
        dec_projected = decoder_state @ proj_dec    # (d_common,)
    else:
        dec_projected = decoder_state

    # Dot product scores: (T_enc, d_common) x (d_common,) -> (T_enc,)
    scores = enc_projected @ dec_projected
    alpha = softmax(scores)

    # Context in original encoder space (not projected space)
    context = alpha @ encoder_states    # (T_enc,) x (T_enc, d_enc) -> (d_enc,)
    return context, alpha

# Demo: reuse the complaint encoder states from before
# encoder_states: shape (8, d_h) where d_h = 16
ctx_dot, alpha_dot = dot_product_attention(encoder_states, decoder_state)

print("Dot product attention demo")
print("=" * 40)
print(f"Encoder states: {encoder_states.shape}")
print(f"Decoder state:  {decoder_state.shape}")
print(f"Context:        {ctx_dot.shape}")
print()
print("Attention weights (dot product) per complaint token:")
for tok, a_b, a_d in zip(found_tokens, alpha_mat[0], alpha_dot):
    bar_b = "#" * int(a_b * 30)
    bar_d = "#" * int(a_d * 30)
    print(f"  {tok:20s}  Bahdanau:{a_b:.3f} {bar_b}")
    print(f"  {'':20s}  DotProd: {a_d:.3f} {bar_d}")
    print()

print("With random weights the patterns will differ.")
print("In a trained model, both should attend to similar tokens for the same task.")

## Lab 2 - Implement Dot Product Attention (Tier 2 - Hard)

**Time**: 25-35 minutes

### Situation

Your Barclays team wants a faster attention layer for a real-time triage API. Bahdanau
attention works but its three learned weight matrices add latency. The encoder and
decoder both produce 16-dim hidden states, so dot product attention is a candidate.

### Task

Implement dot product attention end-to-end in pure NumPy. No learned weights. You decide
the structure of the function body. Reuse `encoder_states` and `decoder_state` from the demo.

### Action

Implement the function from the signature alone. The verification cell expects:
- `context` shape equal to `(d,)` (same dim as encoder hidden states)
- `alpha` shape equal to `(T_enc,)` and summing to 1.0
- Numerical match against the reference `dot_product_attention` (no projection arguments)

### Result

Your context vector and attention weights must match `dot_product_attention` from
the demo. The verification cell will check both.

In [ ]:
# Lab 2: Implement dot product attention end-to-end.
# Fill in the function body. No numbered hints this time -- design the steps yourself.

def my_dot_product_attention(encoder_states, decoder_state):
    """
    Dot product attention with NO learned weights.

    Args:
        encoder_states: shape (T_enc, d) - all encoder hidden states
        decoder_state:  shape (d,) - current decoder hidden state

    Returns:
        context: shape (d,) - weighted sum of encoder states
        alpha:   shape (T_enc,) - attention weights (sum to 1)
    """
    context = None  # YOUR CODE
    alpha = None    # YOUR CODE
    return context, alpha

# Quick test
test_ctx, test_alpha = my_dot_product_attention(encoder_states, decoder_state)
print(f"context shape: {test_ctx.shape if test_ctx is not None else 'None - not implemented yet'}")
print(f"alpha shape:   {test_alpha.shape if test_alpha is not None else 'None - not implemented yet'}")

In [ ]:
# Lab 2 Verification - run after completing my_dot_product_attention.

ref_ctx2, ref_alpha2 = dot_product_attention(encoder_states, decoder_state)
student_ctx2, student_alpha2 = my_dot_product_attention(encoder_states, decoder_state)

all_pass2 = True

if student_ctx2 is None or student_alpha2 is None:
    print("FAIL: Some return values are still None. Complete all three steps.")
    all_pass2 = False
else:
    alpha_sum2 = float(np.sum(student_alpha2))
    if abs(alpha_sum2 - 1.0) < 1e-5:
        print(f"PASS: alpha sums to 1.0 (got {alpha_sum2:.6f})")
    else:
        print(f"FAIL: alpha should sum to 1.0, got {alpha_sum2:.6f}")
        all_pass2 = False

    if np.allclose(student_ctx2, ref_ctx2, atol=1e-5):
        print("PASS: context vector matches reference implementation")
    else:
        print("FAIL: context vector differs from reference")
        all_pass2 = False

    if np.allclose(student_alpha2, ref_alpha2, atol=1e-5):
        print("PASS: attention weights match reference implementation")
    else:
        print("FAIL: attention weights do not match reference")
        all_pass2 = False

if all_pass2:
    print()
    print("All checks passed. Your dot product attention implementation is correct.")

In [ ]:
# Lab 2 safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
if 'my_dot_product_attention' not in dir() or my_dot_product_attention(
        encoder_states, decoder_state)[0] is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    my_dot_product_attention = dot_product_attention

### Stretch (fast finishers)

Compare the attention pattern from `my_dot_product_attention` against your Bahdanau
result for the SAME `encoder_states` and `decoder_state`. Which positions does each
variant focus on? With random (untrained) weights you should expect different patterns.

```python
# Stretch: side-by-side comparison
# _, alpha_b, _ = my_bahdanau_attention(encoder_states, decoder_state, W1, W2, v)
# _, alpha_d    = my_dot_product_attention(encoder_states, decoder_state)
# for tok, ab, ad in zip(found_tokens, alpha_b, alpha_d):
#     print(f"{tok:15s}  Bahdanau:{ab:.3f}  DotProd:{ad:.3f}")
```

### Homework Extension

Generalise `my_dot_product_attention` to accept an optional `temperature` argument.
The temperature divides the scores before softmax (smaller temperature = sharper
attention, larger temperature = flatter). This is a common trick when you want to
control how confident the attention layer is. Verify that with `temperature=1.0` you
recover the original behaviour.

## Section 4 - Scaled Dot Product Attention

There is one more problem with dot product attention at large dimension sizes:
as d_k grows, the dot products grow in magnitude, pushing the softmax into
regions with very small gradients.

The fix: divide by the square root of the key dimension.

    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

This is the exact formula used in the Transformer (Vaswani et al., 2017).

In [ ]:
# Beat 1: Without scaling, large dot products saturate softmax -> near-zero gradients.
# This is a real training failure mode, not a hypothetical.

def show_softmax_saturation():
    """
    Demonstrate that unscaled dot products cause softmax saturation.
    When d_k is large, random Q and K already produce large dot products.
    Softmax of large values gives probabilities very close to 0 or 1.
    Gradient of softmax(x) is s * (1 - s) -- when s -> 0 or 1, gradient -> 0.
    """
    print("Effect of query/key dimension on softmax output range:")
    print(f"{'d_k':>8}  {'max score':>12}  {'max softmax':>14}  {'min softmax':>14}  {'gradient ~':>12}")
    print("-" * 70)

    for d_k in [4, 16, 64, 256, 1024]:
        np.random.seed(42)
        # Simulate one query and 8 keys, all random unit-normal vectors
        q = np.random.randn(d_k)
        K = np.random.randn(8, d_k)

        # Unscaled dot product scores
        scores_unscaled = K @ q   # shape: (8,)
        # Scaled dot product scores
        scores_scaled   = K @ q / np.sqrt(d_k)

        attn_unscaled = softmax(scores_unscaled)
        attn_scaled   = softmax(scores_scaled)

        max_score = float(np.max(np.abs(scores_unscaled)))
        max_prob  = float(np.max(attn_unscaled))
        min_prob  = float(np.min(attn_unscaled))
        max_grad  = float(max_prob * (1 - max_prob))  # derivative of softmax

        print(f"{d_k:>8}  {max_score:>12.2f}  {max_prob:>14.6f}  {min_prob:>14.6f}  {max_grad:>12.6f}")

    print()
    print("At d_k=1024: max softmax is nearly 1.0, min is nearly 0.0.")
    print("Gradient is near zero -> vanishing gradients -> model does not learn.")
    print("Dividing by sqrt(d_k) keeps scores in a reasonable range.")

show_softmax_saturation()

### Beat 2 - Why Scaling Fixes the Saturation Problem

When d_k is large, the dot product Q * K^T grows in variance because it sums d_k
independent random terms. The standard deviation of the raw score grows like sqrt(d_k).

Dividing by sqrt(d_k) normalises the variance back to ~1, keeping softmax inputs in
the range where gradients are meaningful:

    unscaled: score ~ N(0, d_k)     -> softmax saturates at large d_k
    scaled:   score / sqrt(d_k) ~ N(0, 1) -> softmax stays well-conditioned

Intuition: if d_k = 64, a raw dot product score could easily reach 40 or 50.
softmax(50) concentrates almost all probability mass on one position.
After dividing by sqrt(64) = 8 the score is ~5, and softmax remains informative.

This is the only mathematical difference between plain dot product attention and
the scaled version used in the Transformer. Everything else (projection, softmax,
weighted sum) is identical.

In [ ]:
# Beat 3: Scaled dot product attention - the Transformer's core operation.
# Formula: Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V
#
# Q = queries, K = keys, V = values
# In self-attention: Q = K = V = the sequence embeddings
# In cross-attention: Q = decoder, K = V = encoder outputs

def scaled_dot_product_attention(Q, K, V):
    """
    Scaled dot product attention (Vaswani et al., 2017).

    Args:
        Q: queries, shape (batch, T_q, d_k)
        K: keys,    shape (batch, T_k, d_k)
        V: values,  shape (batch, T_k, d_v)

    Returns:
        output:            shape (batch, T_q, d_v) - context vectors
        attention_weights: shape (batch, T_q, T_k) - attention weights
    """
    d_k = Q.shape[-1]   # key dimension - this is what we scale by

    # Step 1: Compute scaled dot product scores.
    # Q K^T: (batch, T_q, d_k) x (batch, d_k, T_k) -> (batch, T_q, T_k)
    # np.matmul broadcasts batch dimension automatically
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)

    # Step 2: Softmax over key positions (axis=-1 = over T_k)
    # Each query attends over all key positions
    attention_weights = softmax(scores, axis=-1)   # (batch, T_q, T_k)

    # Step 3: Weighted sum of values
    # (batch, T_q, T_k) x (batch, T_k, d_v) -> (batch, T_q, d_v)
    output = np.matmul(attention_weights, V)

    return output, attention_weights

# Demo with a batch of 2 complaint sequences, 8 tokens each, d_k = d_v = 16
batch_size = 2
T_seq = 8
d_k = 16

np.random.seed(99)
Q_batch = np.random.randn(batch_size, T_seq, d_k)   # queries
K_batch = np.random.randn(batch_size, T_seq, d_k)   # keys (same as Q for self-attention)
V_batch = np.random.randn(batch_size, T_seq, d_k)   # values (same as Q for self-attention)

output_batch, attn_weights_batch = scaled_dot_product_attention(Q_batch, K_batch, V_batch)

print("Scaled dot product attention demo")
print("=" * 40)
print(f"Q shape: {Q_batch.shape}  -> (batch=2, seq_len=8, d_k=16)")
print(f"K shape: {K_batch.shape}")
print(f"V shape: {V_batch.shape}")
print(f"Output shape: {output_batch.shape}  -> (batch=2, seq_len=8, d_v=16)")
print(f"Attention weights shape: {attn_weights_batch.shape}  -> (batch=2, 8 queries, 8 keys)")
print()
print("Attention weights for batch item 0 (self-attention over 8 complaint tokens):")
print(f"Row sums (must all be 1.0): {attn_weights_batch[0].sum(axis=-1).round(4)}")
print()

# Visualise self-attention pattern for the first sequence
token_labels = found_tokens[:T_seq] if len(found_tokens) >= T_seq else found_tokens + [f"t{i}" for i in range(T_seq - len(found_tokens))]
plot_attention_weight_matrix(attn_weights_batch[0], token_labels, token_labels,
                             title="Scaled Dot Product Self-Attention (random weights)")
print("Diagonal dominance (each token attending mostly to itself) is common with random weights.")
print("Training teaches the model to attend across tokens based on semantic relevance.")

In [ ]:
# Comparison: All three attention variants on the same complaint data.
# Use the real word2vec embeddings for complaint-domain words.

complaint_words = ["unauthorised", "charge", "account", "fraud", "refund", "dispute", "urgent", "help"]
emb, found = get_word2vec_embedding(complaint_words)

print(f"Found {len(found)} words in word2vec: {found}")

# Use word2vec embeddings directly (300-dim) for dot product and scaled dot product
# For Bahdanau we need projection matrices since d_h != 300

d_emb = emb.shape[1]   # 300 (word2vec dimension)

np.random.seed(13)

# ---- Variant 1: Dot product attention (self-attention: Q = K = V = embeddings)
# Must project to a manageable size first
d_proj = 32
proj = np.random.randn(d_emb, d_proj) * 0.1

emb_proj = emb @ proj                           # (N, 32)
Q1 = emb_proj[np.newaxis]                       # (1, N, 32)
K1 = emb_proj[np.newaxis]
V1 = emb_proj[np.newaxis]

# Unscaled
scores_unscaled = np.matmul(Q1, K1.transpose(0, 2, 1))  # (1, N, N)
alpha_unscaled = softmax(scores_unscaled, axis=-1)

# Scaled
_, alpha_scaled = scaled_dot_product_attention(Q1, K1, V1)

# ---- Display
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(alpha_unscaled[0], ax=axes[0], cmap="Blues",
            xticklabels=found, yticklabels=found, annot=True, fmt=".2f")
axes[0].set_title("Dot Product Attention (NOT scaled)")
axes[0].set_xlabel("Key tokens")
axes[0].set_ylabel("Query tokens")

sns.heatmap(alpha_scaled[0], ax=axes[1], cmap="Blues",
            xticklabels=found, yticklabels=found, annot=True, fmt=".2f")
axes[1].set_title("Scaled Dot Product Attention (/ sqrt(d_k))")
axes[1].set_xlabel("Key tokens")
axes[1].set_ylabel("Query tokens")

plt.suptitle("Comparing Unscaled vs Scaled Dot Product Attention\n(random projection, no training)", y=1.02)
plt.tight_layout()
plt.show()

print("With random weights the difference is subtle at d_k=32.")
print("At d_k=512 (Transformer size) the unscaled version saturates severely.")
print("The scaled version remains well-calibrated regardless of d_k.")

### Discussion (3 minutes)

We have now seen three attention variants:
- Bahdanau (additive): learned alignment network, no dimension constraint
- Dot product: simple and fast, requires same dimensions
- Scaled dot product: dot product with 1/sqrt(d_k) normalisation -- used in Transformers

1. From an engineering standpoint at Barclays, which would you choose for a production
   complaint routing system? What are the tradeoffs (speed vs flexibility vs accuracy)?

2. In the scaled dot product formula, V (values) can be different from K (keys).
   In a cross-attention setting (encoder-decoder), Q comes from the decoder and
   K, V come from the encoder. What does this mean conceptually for complaint summarisation?

3. We implemented attention in pure NumPy. What would change if you did this in
   a deep learning framework like PyTorch? Think about gradients, batching, and
   GPU execution. There is an optional companion notebook that ports this exact
   code to PyTorch if you want to check your answer.

## Lab 3 - Implement Scaled Dot Product Attention (Tier 1 - Guided)

**Time**: 15-20 minutes

### Situation

The Barclays research team is prototyping a Transformer-based complaint summariser.
Before plugging in PyTorch, your lead wants you to demonstrate that you understand
the core operation of the Transformer: scaled dot product attention. Get this right
and the rest of the architecture is mostly bookkeeping.

### Task

Implement `my_scaled_dot_product_attention(Q, K, V)` in pure NumPy. Inputs are batched
3D arrays: `(batch, T_q, d_k)` for Q, `(batch, T_k, d_k)` for K, `(batch, T_k, d_v)` for V.

### Action

Complete the four numbered steps below. Each step is one line of NumPy.

1. Compute the raw dot product scores: `Q @ K^T`.
2. Scale by `sqrt(d_k)`.
3. Apply `softmax` along the key axis (`axis=-1`).
4. Weighted sum of values: `attention_weights @ V`.

### Result

Your function must match `scaled_dot_product_attention` from the demo on the same
`Q_batch`, `K_batch`, `V_batch` inputs. The verification cell checks both outputs.

In [ ]:
# Lab 3: Implement scaled dot product attention.
# Complete the four numbered steps. Reference: scaled_dot_product_attention from the demo.

my_scaled_dot_product_attention = None  # YOUR CODE

def _lab3_template(Q, K, V):
    """
    Scaled dot product attention.

    Args:
        Q: shape (batch, T_q, d_k)
        K: shape (batch, T_k, d_k)
        V: shape (batch, T_k, d_v)

    Returns:
        output:            shape (batch, T_q, d_v)
        attention_weights: shape (batch, T_q, T_k)
    """
    d_k = Q.shape[-1]

    # Step 1: Raw dot product scores. Hint: np.matmul(Q, K.transpose(0, 2, 1)).
    raw_scores = None  # YOUR CODE

    # Step 2: Scale by sqrt(d_k).
    scaled_scores = None  # YOUR CODE

    # Step 3: Softmax over the key axis (axis=-1).
    attention_weights = None  # YOUR CODE

    # Step 4: Weighted sum of values.
    output = None  # YOUR CODE

    return output, attention_weights

# When you have filled in the four steps above, assign the function to the lab variable:
# my_scaled_dot_product_attention = _lab3_template

# Quick check (will print None until you assign)
print(f"my_scaled_dot_product_attention is: {my_scaled_dot_product_attention}")

In [ ]:
# Lab 3 Verification - run after completing my_scaled_dot_product_attention above.

if my_scaled_dot_product_attention is None:
    print("FAIL: my_scaled_dot_product_attention is still None. Assign it to your function.")
else:
    ref_out, ref_attn = scaled_dot_product_attention(Q_batch, K_batch, V_batch)
    student_out, student_attn = my_scaled_dot_product_attention(Q_batch, K_batch, V_batch)

    all_pass3 = True

    # Shape checks
    if student_out.shape == ref_out.shape:
        print(f"PASS: output shape matches {student_out.shape}")
    else:
        print(f"FAIL: output shape {student_out.shape} != reference {ref_out.shape}")
        all_pass3 = False

    # Row-sum check
    row_sums = student_attn.sum(axis=-1)
    if np.allclose(row_sums, 1.0, atol=1e-5):
        print("PASS: attention weights sum to 1.0 along key axis")
    else:
        print(f"FAIL: attention weight rows do not sum to 1.0 (got {row_sums})")
        all_pass3 = False

    # Numeric match
    if np.allclose(student_out, ref_out, atol=1e-5):
        print("PASS: output matches reference implementation")
    else:
        max_diff = float(np.max(np.abs(student_out - ref_out)))
        print(f"FAIL: output differs from reference (max diff: {max_diff:.6f})")
        all_pass3 = False

    if np.allclose(student_attn, ref_attn, atol=1e-5):
        print("PASS: attention weights match reference implementation")
    else:
        print("FAIL: attention weights do not match reference")
        all_pass3 = False

    if all_pass3:
        print()
        print("All checks passed. Your scaled dot product attention is correct.")

In [ ]:
# Lab 3 safety-net: run this if you did not finish Lab 3.
# SKIP this cell if you DID finish Lab 3.
if my_scaled_dot_product_attention is None:
    print("Using Lab 3 safety-net so the rest of the notebook can run.")
    my_scaled_dot_product_attention = scaled_dot_product_attention

### Stretch (fast finishers)

Extend `my_scaled_dot_product_attention` with an optional `mask` argument of shape
`(batch, T_q, T_k)`. Wherever `mask` is 0 (or False), set the corresponding raw score
to a very large negative number (e.g. `-1e9`) BEFORE the softmax. This is exactly how
the Transformer hides padding positions and prevents the decoder from attending to
future tokens (causal masking).

Verify on a 2-sequence batch where the second sequence is padded: the attention
weights on padded positions should be effectively zero after softmax.

### Homework Extension

Generalise the masked version to support both padding masks AND causal masks at once.
Then visualise the resulting attention weights for a 6-token sequence with a causal
mask: the heatmap should be lower-triangular.

## Wrap-Up

### Key takeaways

1. The seq2seq fixed bottleneck loses information from long inputs. The further a
   token is from the end of a sequence, the more it gets diluted in the context
   vector.

2. Bahdanau attention solves this by computing a DIFFERENT context vector at each
   decoder step, attending selectively to encoder hidden states via learned
   alignment weights.

3. Dot product attention is a simpler variant that requires the same dimensions
   for queries and keys. It is faster but less flexible than Bahdanau.

4. Scaled dot product attention divides scores by sqrt(d_k) to prevent softmax
   saturation and vanishing gradients at large key dimensions.

5. Attention weights are always non-negative and sum to 1 along the key dimension.
   You can visualise them as a heatmap to understand what the model focuses on.

### Connecting this back to using models

Remember the three user-facing facts from the start of this notebook: context
windows, long-input quality, and the square-with-length cost of long prompts.
You have now seen the mechanism behind all three. The "look back at any position"
ability is the weighted sum over encoder states. The cost growth is the dot
product of every query against every key.

### Where to go next (optional)

This notebook is the end of the pure-NumPy attention deep-dive. There is an
optional companion notebook that ports these exact mechanisms to PyTorch. It
shows how PyTorch's nn.Module wraps the same math into a trainable class, what
torch.nn.functional.scaled_dot_product_attention looks like, and how to apply
attention to a small triage task with automatic gradients.

That companion is optional, not a required next step. The attention concept you
need as a user is taught in the main course path; this notebook and its PyTorch
companion are the from-scratch builds for learners who want the internals. The
math is identical to what you built here; the framework only adds automatic
differentiation and GPU execution. Read it if you are curious; skip it if you
have what you came for.

## Homework Extensions

These are designed for 30-60 minutes of async study after class.

In [ ]:
# Homework Extension 1:
# Implement Bahdanau attention from the equation alone.
# No scaffold, no step comments. Just the signature and docstring.
# Verify your result matches bahdanau_attention() from the demo.

def bahdanau_from_equation(encoder_states, decoder_state, W1, W2, v):
    """
    Implement Bahdanau attention from this equation:

        e_{t,i} = v^T * tanh(W1 h_i + W2 s_{t-1})   for i = 1..T
        alpha   = softmax(e)
        c_t     = sum_i(alpha_i * h_i)

    Args:
        encoder_states: numpy array (T, d_h)
        decoder_state:  numpy array (d_s,)
        W1:             numpy array (d_align, d_h)
        W2:             numpy array (d_align, d_s)
        v:              numpy array (d_align,)

    Returns:
        context_vector: numpy array (d_h,)
        alpha:          numpy array (T,) - must sum to 1
    """
    pass  # YOUR IMPLEMENTATION (no scaffold, no hints)

# Verification (run after implementing):
# ref_ctx, ref_alpha, _ = bahdanau_attention(encoder_states, decoder_state, W1, W2, v)
# hw_ctx, hw_alpha = bahdanau_from_equation(encoder_states, decoder_state, W1, W2, v)
# assert np.allclose(hw_ctx, ref_ctx, atol=1e-5), "Context vector does not match"
# assert np.allclose(hw_alpha, ref_alpha, atol=1e-5), "Alpha does not match"
# print("Homework 1 verified correctly.")

In [ ]:
# Homework Extension 2:
# Extend your implementation to handle a batch of decoder states simultaneously.
# This is the production-relevant version: processing multiple complaints at once.

def bahdanau_attention_batched(encoder_states, decoder_states, W1, W2, v):
    """
    Compute Bahdanau attention for multiple decoder steps at once.

    Args:
        encoder_states: numpy array (T_enc, d_h)
        decoder_states: numpy array (T_dec, d_s)
        W1, W2, v:      alignment parameters (same shapes as before)

    Returns:
        context_vectors: numpy array (T_dec, d_h)
        alpha_matrix:    numpy array (T_dec, T_enc)

    Hint: look at bahdanau_attention_vectorised from the demo.
    Try implementing it WITHOUT looking at that cell.
    """
    pass  # YOUR IMPLEMENTATION

# Verification:
# T_dec_test = 4
# decoder_states_test = np.random.randn(T_dec_test, d_s)
# ref_ctxs, ref_alphas = bahdanau_attention_vectorised(encoder_states, decoder_states_test, W1, W2, v)
# hw_ctxs, hw_alphas = bahdanau_attention_batched(encoder_states, decoder_states_test, W1, W2, v)
# assert np.allclose(hw_ctxs, ref_ctxs, atol=1e-5), "Context vectors do not match"
# print("Homework 2 verified correctly.")

*End of the optional deep-dive: How Attention Works (Pure Python)*

This was an optional notebook. The main course path continues independently of
it. If you want the framework version of everything you built here, the optional
PyTorch companion notebook is available, but it is not required.